# 1. Libraries and Data Loading

In [64]:
import numpy as np
import pandas as pd

import sklearn

import plotly.express as px

import os

In [65]:
# Ensure statsmodels is installed in THIS kernel (used by the ARIMA model,
# section 8). %pip installs into the running kernel's environment; we only
# install if missing, then import so the rest of the notebook can rely on it.
try:
    import statsmodels  # noqa: F401
except ModuleNotFoundError:
    %pip install -q statsmodels
    import statsmodels  # noqa: F401

print('statsmodels', statsmodels.__version__)


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
statsmodels 0.14.6


In [66]:
data = pd.read_csv("../../data/data_raw.csv")
data.head(5)

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories


In [67]:
data.dtypes

week                  str
sku                 int64
weekly_sales      float64
feat_main_page       bool
color                 str
price             float64
vendor              int64
functionality         str
dtype: object

In [68]:
data['week']=pd.to_datetime(data['week'])

In [69]:
data.sort_values(['sku','week'])

data['week'].min(), data['week'].max()

(Timestamp('2016-10-31 00:00:00'), Timestamp('2018-09-24 00:00:00'))

In [70]:
def panel_data_split(data: pd.DataFrame, val_cutoff: int, test_cutoff: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    '''
    Splits panel data into train/val/test splits based on cutoff week thresholds.
    -----------
    Parameters:
    -----------
    data: panel data with 'week' column
    val_cutoff: number of weeks to use for validation set
    test_cutoff: number of weeks to use for test set

    Returns:
    ---------
    train: training observations, data subset
    val: validation set
    test: test set
    '''
    #sort the data
    data=data.sort_values(['sku','week'])

    #get the number of weeks
    unique_weeks=sorted(data['week'].unique())
    #find index that cutsoff for both test and val
    test_idx = len(unique_weeks) - test_cutoff
    val_idx = test_idx - val_cutoff

    # get date time by index
    test_start_date = unique_weeks[test_idx]
    val_start_date = unique_weeks[val_idx]

    #boolean mask filtering
    train_mask = data['week'] < val_start_date
    val_mask = (data['week'] >= val_start_date) & (data['week'] < test_start_date)
    test_mask = data['week'] >= test_start_date
    #store in train,val,test variables
    train = data[train_mask].copy()
    val = data[val_mask].copy()
    test = data[test_mask].copy()

    return train, val, test


In [71]:
train, val, test = panel_data_split(data, val_cutoff=13, test_cutoff=13)

# 2. EDA & Preprocessing

## 2.1 Summary Stats

In [72]:
train.info()

<class 'pandas.DataFrame'>
Index: 3256 entries, 0 to 4373
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   week            3256 non-null   datetime64[us]
 1   sku             3256 non-null   int64         
 2   weekly_sales    3256 non-null   float64       
 3   feat_main_page  3256 non-null   bool          
 4   color           3251 non-null   str           
 5   price           3256 non-null   float64       
 6   vendor          3256 non-null   int64         
 7   functionality   3256 non-null   str           
dtypes: bool(1), datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 206.7 KB


In [73]:
train.describe()

,week,sku,weekly_sales,price,vendor
count,3256,3256.000000,3256.000000,3256.000000,3256.000000
mean,2017-07-13 12:00:00,22.500000,79.303747,43.840660,6.909091
min,2016-10-31 00:00:00,1.000000,1.000000,2.390000,1.000000
25%,2017-03-06 00:00:00,11.750000,10.000000,15.357500,6.000000
50%,2017-07-13 12:00:00,22.500000,23.000000,27.490000,6.500000
75%,2017-11-20 00:00:00,33.250000,72.000000,54.990000,9.000000
max,2018-03-26 00:00:00,44.000000,7512.000000,204.460000,10.000000
std,NaN,12.700376,260.067332,41.831612,2.503275


### 2.2 Visualisations

In [74]:
train.isnull().sum()

week              0
sku               0
weekly_sales      0
feat_main_page    0
color             5
price             0
vendor            0
functionality     0
dtype: int64

In [75]:
train[train.isna().any(axis=1)]

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality
802,2016-11-14,9,54.0,True,NaN,139.44,9,11.Fitness trackers
803,2016-11-21,9,71.0,True,NaN,141.16,9,11.Fitness trackers
4133,2017-06-19,42,4.0,False,NaN,27.33,10,09.Smartphone stands
4200,2016-10-31,43,5.0,True,NaN,109.99,9,11.Fitness trackers
4314,2017-02-06,44,5.0,False,NaN,53.99,6,09.Smartphone stands


In [76]:
train['color'].unique()

<StringArray>
[ 'black',   'blue', 'purple',    'red',      nan,  'white',   'none',
  'green',   'grey',   'gold',   'pink']
Length: 11, dtype: str

In [77]:
train['color'].value_counts()

color
black     1254
blue       518
red        370
green      296
grey       222
white      148
none       148
gold       147
purple      74
pink        74
Name: count, dtype: int64

In [78]:
train['color'].fillna('none', inplace=True)

/var/folders/8p/3y0vt8nx0j77_stq2p7d95t80000gn/T/ipykernel_2850/3656370375.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  train['color'].fillna('none', inplace=True)


0       black
1       black
2       black
3       black
4       black
        ...  
4369    black
4370    black
4371    black
4372    black
4373    black
Name: color, Length: 3256, dtype: str

In [79]:
for index,row in train.iterrows():
    print(row)

week                      2016-10-31 00:00:00
sku                                         1
weekly_sales                            135.0
feat_main_page                           True
color                                   black
price                                   10.16
vendor                                      6
functionality     06.Mobile phone accessories
Name: 0, dtype: object
week                      2016-11-07 00:00:00
sku                                         1
weekly_sales                            102.0
feat_main_page                           True
color                                   black
price                                    9.86
vendor                                      6
functionality     06.Mobile phone accessories
Name: 1, dtype: object
week                      2016-11-14 00:00:00
sku                                         1
weekly_sales                            110.0
feat_main_page                           True
color                             

# 3. Feature Engineering

In [80]:
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories


## 3.1 Sales Lag

In [81]:
train['1week_sales_lag']=train.sort_values('week').groupby('sku')['weekly_sales'].shift(1)
train['2week_sales_lag']=train.sort_values('week').groupby('sku')['weekly_sales'].shift(2)
train['3week_sales_lag']=train.sort_values('week').groupby('sku')['weekly_sales'].shift(3)
train['4week_sales_lag']=train.sort_values('week').groupby('sku')['weekly_sales'].shift(4)

## 3.2 Revenue

In [82]:
# Lagged revenue uses only PRIOR weeks (past price x past sales) -> valid feature.
# Current-week weekly_revenue = price x weekly_sales contains the target
# (weekly_sales) -> leakage. Compute it only to derive the lags, then drop it.
train['weekly_revenue'] = train['price'] * train['weekly_sales']

train['1week_revenue_lag'] = train.sort_values('week').groupby('sku')['weekly_revenue'].shift(1)
train['2week_revenue_lag'] = train.sort_values('week').groupby('sku')['weekly_revenue'].shift(2)
train['3week_revenue_lag'] = train.sort_values('week').groupby('sku')['weekly_revenue'].shift(3)
train['4week_revenue_lag'] = train.sort_values('week').groupby('sku')['weekly_revenue'].shift(4)

train = train.drop(columns='weekly_revenue')


In [83]:
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,3week_sales_lag,4week_sales_lag,1week_revenue_lag,2week_revenue_lag,3week_revenue_lag,4week_revenue_lag
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,NaN,NaN,1371.60,NaN,NaN,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,NaN,NaN,1005.72,1371.60,NaN,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,135.0,NaN,1126.40,1005.72,1371.60,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,102.0,135.0,1050.29,1126.40,1005.72,1371.6


## 3.3 Leakage Removal (drop target-derived features)

In [84]:
# Leakage removal. weekly_revenue (= price x weekly_sales) and the
# sales/revenue %_change columns all embed the current-week target
# (weekly_sales): a model fed these can trivially recover the target, giving
# inflated test metrics and a useless production model.
# Kept: *_lag features (prior weeks only) and price %_change (price is known
# in advance). Dropped here, and made idempotent so re-running earlier
# experiment cells can't silently leave these behind.
leaky_cols = (
    ['weekly_revenue']
    + [f'{w}week_sales_%_change' for w in range(1, 5)]
    + [f'{w}week_revenue_%_change' for w in range(1, 5)]
)
train = train.drop(columns=[c for c in leaky_cols if c in train.columns])
train.head()


,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,3week_sales_lag,4week_sales_lag,1week_revenue_lag,2week_revenue_lag,3week_revenue_lag,4week_revenue_lag
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,NaN,NaN,1371.60,NaN,NaN,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,NaN,NaN,1005.72,1371.60,NaN,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,135.0,NaN,1126.40,1005.72,1371.60,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,102.0,135.0,1050.29,1126.40,1005.72,1371.6


In [85]:
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,3week_sales_lag,4week_sales_lag,1week_revenue_lag,2week_revenue_lag,3week_revenue_lag,4week_revenue_lag
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,NaN,NaN,1371.60,NaN,NaN,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,NaN,NaN,1005.72,1371.60,NaN,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,135.0,NaN,1126.40,1005.72,1371.60,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,102.0,135.0,1050.29,1126.40,1005.72,1371.6


## 3.4 Price

In [86]:
def add_lag_features(data: pd.DataFrame, col: str, group_key: str = 'sku',
                     week_col: str = 'week', max_lag: int = 4) -> pd.DataFrame:
    '''
    Build 1..max_lag week lag features for `col` within each `group_key` series.

    Also adds, once per call:
      - `week_index`: 0-based position of each row within its series (time-ordered),
        i.e. how many prior weeks of history exist for that row.
      - `is_warmup`: True for the first `max_lag` rows of every series. These are
        the rows whose lag columns are NaN by construction (no prior week to
        shift from) and should be dropped before modelling.

    Series identity is `group_key` (sku) only - consistent across the sales,
    revenue and price lag features. Sorting by week is required so that `.shift`
    and `.cumcount` are ordered in time, not in raw row order.
    '''
    data = data.sort_values([group_key, week_col]).copy()

    grp = data.groupby(group_key)[col]
    for lag in range(1, max_lag + 1):
        data[f'{lag}week_{col}_lag'] = grp.shift(lag)

    data['week_index'] = data.groupby(group_key).cumcount()
    data['is_warmup'] = data['week_index'] < max_lag

    return data


# Price lags + warmup flag. add_lag_features is generic - the sales/revenue
# lag cells above can be replaced with the same call if desired.
train = add_lag_features(train, 'price', group_key='sku', max_lag=4)
train.head()


,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,...,1week_revenue_lag,2week_revenue_lag,3week_revenue_lag,4week_revenue_lag,1week_price_lag,2week_price_lag,3week_price_lag,4week_price_lag,week_index,is_warmup
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,True
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,...,1371.60,NaN,NaN,NaN,10.16,NaN,NaN,NaN,1,True
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,...,1005.72,1371.60,NaN,NaN,9.86,10.16,NaN,NaN,2,True
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,...,1126.40,1005.72,1371.60,NaN,10.24,9.86,10.16,NaN,3,True
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,...,1050.29,1126.40,1005.72,1371.6,8.27,10.24,9.86,10.16,4,False


In [87]:
train['1week_price_%_change']=round((train['price']-train['1week_price_lag'])/train['1week_price_lag'],2)
train['2week_price_%_change']=round((train['price']-train['2week_price_lag'])/train['2week_price_lag'],2)
train['3week_price_%_change']=round((train['price']-train['3week_price_lag'])/train['3week_price_lag'],2)
train['4week_price_%_change']=round((train['price']-train['4week_price_lag'])/train['4week_price_lag'],2)
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,...,1week_price_lag,2week_price_lag,3week_price_lag,4week_price_lag,week_index,is_warmup,1week_price_%_change,2week_price_%_change,3week_price_%_change,4week_price_%_change
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,...,NaN,NaN,NaN,NaN,0,True,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,...,10.16,NaN,NaN,NaN,1,True,-0.03,NaN,NaN,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,...,9.86,10.16,NaN,NaN,2,True,0.04,0.01,NaN,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,...,10.24,9.86,10.16,NaN,3,True,-0.19,-0.16,-0.19,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,...,8.27,10.24,9.86,10.16,4,False,0.07,-0.14,-0.10,-0.13


## 3.5 Promotion, Calendar, Rolling, Price & Trend Features (Tier 1 & 2)

All trailing windows are past-only (`.shift(1)` before `.rolling`) and series
are keyed by `sku`. Run before the `is_warmup` drop so windows see full history.

In [88]:
def add_promo_features(data, promo_col='feat_main_page', price_change_col='1week_price_%_change',
                       group_key='sku', week_col='week', lags=(1, 2, 4), window=4):
    """Promotion signals from `promo_col` (known in advance, so the current week
    is a legitimate feature).

    Adds, per `group_key` series in week order:
      - `{promo_col}_lag{n}`    : promo flag n weeks ago, for n in `lags`
                                  (float 0/1 so it stays model-ready)
      - `weeks_since_promo`     : weeks since the most recent promo (0 on a promo
                                  week; NaN before the first ever promo)
      - `promo_count_{window}w` : promos in the trailing `window` weeks (past only)
      - `promo_x_price_change`  : promo x prior-week price %-change interaction
    """
    data = data.sort_values([group_key, week_col]).copy()
    g = data.groupby(group_key)[promo_col]

    for n in lags:
        data[f'{promo_col}_lag{n}'] = g.shift(n).astype(float)

    def _since(flag):
        blocks = flag.cumsum()
        return flag.groupby(blocks).cumcount().mask(blocks == 0)

    data['weeks_since_promo'] = g.transform(_since)
    data[f'promo_count_{window}w'] = g.transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).sum())
    data['promo_x_price_change'] = data[promo_col].astype(int) * data[price_change_col]
    return data


def add_calendar_features(data, week_col='week'):
    """Calendar features from the datetime `week_col`: week-of-year, month,
    quarter, an is-Q4 (holiday-season) flag, and a cyclic sin/cos encoding of
    week-of-year for linear models. (Lifecycle trend already exists as
    `week_index` from add_lag_features.)
    """
    data = data.copy()
    wk = data[week_col].dt
    iso_week = wk.isocalendar().week.astype(int)
    data['week_of_year'] = iso_week
    data['month'] = wk.month
    data['quarter'] = wk.quarter
    data['is_q4'] = wk.quarter.eq(4).astype(int)
    data['week_of_year_sin'] = np.sin(2 * np.pi * iso_week / 52)
    data['week_of_year_cos'] = np.cos(2 * np.pi * iso_week / 52)
    return data


def add_rolling_sales_features(data, target='weekly_sales', group_key='sku',
                               week_col='week', windows=(4, 8), std_window=4):
    """Trailing (past-only) rolling stats of `target`. Each window applies
    `.shift(1)` before `.rolling`, so only weeks strictly before the current
    row enter the window (no leakage).

    Adds `sales_roll_mean_{w}w` for w in `windows` and
    `sales_roll_std_{std_window}w` (volatility; reused later for residual CIs).
    """
    data = data.sort_values([group_key, week_col]).copy()
    g = data.groupby(group_key)[target]
    for w in windows:
        data[f'sales_roll_mean_{w}w'] = g.transform(
            lambda s, w=w: s.shift(1).rolling(w, min_periods=1).mean())
    data[f'sales_roll_std_{std_window}w'] = g.transform(
        lambda s: s.shift(1).rolling(std_window, min_periods=2).std())
    return data


def add_price_features(data, price_col='price', group_key='sku', week_col='week',
                       mean_window=8, discount_ratio=0.95, sku_mean=None):
    """Price-shape features (price is known in advance).

    Adds:
      - `log_price`         : log price (elasticity is multiplicative)
      - `price_vs_sku_mean` : (price - reference sku mean price) / reference
      - `discount_flag`     : price below `discount_ratio` x trailing
                              `mean_window`-week mean price

    `sku_mean`: optional Series mapping `group_key` -> mean price, fit on TRAIN.
    Pass it for val/test so `price_vs_sku_mean` uses train-derived means (no
    leakage). If None, the per-sku mean is computed from THIS frame, which is
    correct for the train split itself.
    """
    data = data.sort_values([group_key, week_col]).copy()
    if sku_mean is None:
        ref_mean = data.groupby(group_key)[price_col].transform('mean')
    else:
        ref_mean = data[group_key].map(sku_mean)
    data['log_price'] = np.log(data[price_col])
    data['price_vs_sku_mean'] = (data[price_col] - ref_mean) / ref_mean
    trailing_mean = data.groupby(group_key)[price_col].transform(
        lambda s: s.shift(1).rolling(mean_window, min_periods=1).mean())
    data['discount_flag'] = (data[price_col] < discount_ratio * trailing_mean).astype(int)
    return data


def add_trend_features(data, target='weekly_sales', group_key='sku', week_col='week',
                       short=4, long=13):
    """Short-vs-long demand trend: trailing `short`-week mean minus trailing
    `long`-week mean of `target` (both past-only via `.shift(1)`).

    Adds `sales_trend_{short}w_vs_{long}w`.
    """
    data = data.sort_values([group_key, week_col]).copy()
    g = data.groupby(group_key)[target]
    short_mean = g.transform(lambda s: s.shift(1).rolling(short, min_periods=1).mean())
    long_mean = g.transform(lambda s: s.shift(1).rolling(long, min_periods=1).mean())
    data[f'sales_trend_{short}w_vs_{long}w'] = short_mean - long_mean
    return data


In [89]:
# Tier 1 & 2 features. Run BEFORE the is_warmup drop so trailing windows see
# full history; the drop then removes the early NaN (warm-up) rows.
train = add_promo_features(train)
train = add_calendar_features(train)
train = add_rolling_sales_features(train)
train = add_price_features(train)
train = add_trend_features(train)
train.head()


,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,...,is_q4,week_of_year_sin,week_of_year_cos,sales_roll_mean_4w,sales_roll_mean_8w,sales_roll_std_4w,log_price,price_vs_sku_mean,discount_flag,sales_trend_4w_vs_13w
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,...,1,-0.822984,0.568065,NaN,NaN,NaN,2.318458,-0.560060,0,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,...,1,-0.748511,0.663123,135.000000,135.000000,NaN,2.288486,-0.573050,0,0.0
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,...,1,-0.663123,0.748511,118.500000,118.500000,23.334524,2.326302,-0.556596,0,0.0
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,...,1,-0.568065,0.822984,115.666667,115.666667,17.214335,2.112635,-0.641899,1,0.0
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,...,1,-0.464723,0.885456,118.500000,118.500000,15.154757,2.178155,-0.617651,1,0.0


## 3.5b Log Target & Promo Flag

`add_log_target` keeps `weekly_sales` (level) and adds `log_weekly_sales` so a
log target can be compared at modelling time. `add_is_promo` adds an explicit
0/1 `is_promo` flag for the current promotion week. Both are separate,
independently-called functions.

In [90]:
def add_log_target(data, target='weekly_sales', out='log_weekly_sales'):
    """Add a log-scale copy of the target while keeping the level target.

    Weekly demand is right-skewed and multiplicative, so modelling log sales
    stabilises variance and makes errors proportional. np.log1p keeps any
    zero-sales weeks finite (here min sales is 1, so log1p ~ log). Model
    selection later can compare the level vs log target.
    """
    data = data.copy()
    data[out] = np.log1p(data[target])
    return data


def add_is_promo(data, promo_col='feat_main_page', out='is_promo'):
    """Explicit 0/1 promotion flag for the current week.

    `feat_main_page` is boolean and known in advance; `is_promo` is its integer
    form so linear models, interactions and metrics treat it as a numeric
    promo indicator capturing the in-week promotional effect.
    """
    data = data.copy()
    data[out] = data[promo_col].astype(int)
    return data


In [91]:
train = add_log_target(train)
train = add_is_promo(train)
train.head()


,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,...,week_of_year_cos,sales_roll_mean_4w,sales_roll_mean_8w,sales_roll_std_4w,log_price,price_vs_sku_mean,discount_flag,sales_trend_4w_vs_13w,log_weekly_sales,is_promo
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,...,0.568065,NaN,NaN,NaN,2.318458,-0.560060,0,NaN,4.912655,1
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,...,0.663123,135.000000,135.000000,NaN,2.288486,-0.573050,0,0.0,4.634729,1
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,135.0,...,0.748511,118.500000,118.500000,23.334524,2.326302,-0.556596,0,0.0,4.709530,1
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,102.0,...,0.822984,115.666667,115.666667,17.214335,2.112635,-0.641899,1,0.0,4.852030,1
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,...,0.885456,118.500000,118.500000,15.154757,2.178155,-0.617651,1,0.0,4.442651,1


## 3.6 Warm-up Row Drop (verify is_warmup, drop early NaN rows)

In [92]:
# 'is_warmup' (set by add_lag_features) marks the first max_lag weeks of every
# sku - exactly the rows whose lag columns are NaN by construction. Verify the
# flag lines up with the NaN lag rows, then drop them before modelling.
mismatches = (train['is_warmup'] != train['4week_price_lag'].isna()).sum()
print(f"is_warmup vs NaN 4week_price_lag mismatches: {mismatches}")

before = len(train)
train = train[~train['is_warmup']].copy()
print(f"Dropped {before - len(train)} warmup rows; {len(train)} rows remain")
train.head(20)


is_warmup vs NaN 4week_price_lag mismatches: 0
Dropped 176 warmup rows; 3080 rows remain


,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,2week_sales_lag,...,week_of_year_cos,sales_roll_mean_4w,sales_roll_mean_8w,sales_roll_std_4w,log_price,price_vs_sku_mean,discount_flag,sales_trend_4w_vs_13w,log_weekly_sales,is_promo
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,110.0,...,8.854560e-01,118.50,118.500000,15.154757,2.178155,-0.617651,1,0.000000,4.442651,1
5,2016-12-05,1,87.0,True,black,8.98,6,06.Mobile phone accessories,84.0,127.0,...,9.350162e-01,105.75,111.600000,17.858238,2.195000,-0.611155,1,-5.850000,4.477337,1
6,2016-12-12,1,64.0,True,black,10.40,6,06.Mobile phone accessories,87.0,84.0,...,9.709418e-01,102.00,107.500000,20.314199,2.341806,-0.549668,0,-5.500000,4.174387,1
7,2016-12-19,1,128.0,True,black,9.57,6,06.Mobile phone accessories,64.0,87.0,...,9.927089e-01,90.50,101.285714,26.388129,2.258633,-0.585608,0,-10.785714,4.859812,1
8,2016-12-26,1,23.0,True,black,10.49,6,06.Mobile phone accessories,128.0,64.0,...,1.000000e+00,90.75,104.625000,26.849891,2.350422,-0.545771,0,-13.875000,3.178054,1
9,2017-01-02,1,154.0,True,black,9.19,6,06.Mobile phone accessories,23.0,128.0,...,9.927089e-01,75.50,90.625000,43.882419,2.218116,-0.602062,0,-20.055556,5.043425,1
10,2017-01-09,1,85.0,True,black,11.91,6,06.Mobile phone accessories,154.0,23.0,...,9.709418e-01,92.25,97.125000,59.679002,2.477378,-0.484283,0,-9.150000,4.454347,1
11,2017-01-16,1,11.0,False,black,23.19,6,06.Mobile phone accessories,85.0,154.0,...,9.350162e-01,97.50,94.000000,57.239264,3.143721,0.004155,0,-2.409091,2.484907,0
12,2017-01-23,1,19.0,False,black,20.59,6,06.Mobile phone accessories,11.0,85.0,...,8.854560e-01,68.25,79.500000,65.723537,3.024806,-0.108429,0,-24.250000,2.995732,0
13,2017-01-30,1,17.0,False,black,20.74,6,06.Mobile phone accessories,19.0,11.0,...,8.229839e-01,67.25,71.375000,66.665208,3.032064,-0.101933,0,-19.596154,2.890372,0


## 3.7 Apply Full Pipeline to Val & Test (carried history, no leakage)

`val`/`test` are rebuilt from raw `data` with prior weeks carried in so
boundary lags/rolling windows are correct, then sliced back to their own
weeks. `price_vs_sku_mean` uses train-fitted per-sku means. Same separate
feature functions as train - no monolithic wrapper.

In [93]:
# Split boundaries (same logic & cutoffs as panel_data_split: val=13, test=13).
weeks = sorted(data['week'].unique())
test_start, val_start = weeks[-13], weeks[-26]

# Train-fitted per-sku mean price -> passed to add_price_features so val/test
# 'price_vs_sku_mean' uses TRAIN means only (no leakage).
train_sku_price_mean = train.groupby('sku')['price'].mean()


def engineer_split(raw, keep_from):
    """Build the full feature set for one split.

    `raw` already includes carried prior history so boundary lags/rolling
    windows are correct; rows before `keep_from` are sliced off afterwards.
    This only does the concat/slice that carrying history requires - every
    feature function below is still the same separate, independently-called
    function used for train (no monolithic feature wrapper).
    """
    df = raw.sort_values(['sku', 'week']).copy()
    df['color'] = df['color'].fillna('none')

    # 3.1 sales lags
    for n in range(1, 5):
        df[f'{n}week_sales_lag'] = df.groupby('sku')['weekly_sales'].shift(n)

    # 3.2 revenue lags (current-week weekly_revenue is leakage -> dropped)
    df['weekly_revenue'] = df['price'] * df['weekly_sales']
    for n in range(1, 5):
        df[f'{n}week_revenue_lag'] = df.groupby('sku')['weekly_revenue'].shift(n)
    df = df.drop(columns='weekly_revenue')

    # 3.4 price lags + week_index + is_warmup, then price %_change
    df = add_lag_features(df, 'price')
    for n in range(1, 5):
        df[f'{n}week_price_%_change'] = round(
            (df['price'] - df[f'{n}week_price_lag']) / df[f'{n}week_price_lag'], 2)

    # 3.5 / 3.5b - Tier 1 & 2, log target, promo flag
    df = add_promo_features(df)
    df = add_calendar_features(df)
    df = add_rolling_sales_features(df)
    df = add_price_features(df, sku_mean=train_sku_price_mean)
    df = add_trend_features(df)
    df = add_log_target(df)
    df = add_is_promo(df)

    # keep only this split's own weeks, then drop warm-up rows
    df = df[df['week'] >= keep_from].copy()
    return df[~df['is_warmup']].copy()


# val carries all train history; test carries train+val history.
val = engineer_split(data[data['week'] < test_start], val_start)
test = engineer_split(data.copy(), test_start)

for name, d in [('train', train), ('val', val), ('test', test)]:
    mm = (d['is_warmup'] != d['4week_price_lag'].isna()).sum()
    print(f"{name:5s}: {len(d):5d} rows, {d.shape[1]} cols, "
          f"is_warmup/NaN-lag mismatches={mm}")


train:  3080 rows, 47 cols, is_warmup/NaN-lag mismatches=0
val  :   572 rows, 47 cols, is_warmup/NaN-lag mismatches=0
test :   572 rows, 47 cols, is_warmup/NaN-lag mismatches=0


# 4. Categorical Encoding

One-hot encode the categorical attributes (`color`, `vendor`,
`functionality`). The encoder is **fit on train only** and reused to transform
val/test, so all splits share an identical column set and no val/test category
information leaks into the fitted vocabulary. `vendor` is treated as a category
(its integer codes are nominal, not ordinal).

In [94]:
from sklearn.preprocessing import OneHotEncoder

CATEGORICAL_COLS = ['color', 'vendor', 'functionality']


def one_hot_encode(data, cols=CATEGORICAL_COLS, encoder=None):
    """One-hot encode `cols`, dropping the originals.

    Fit a fresh encoder when `encoder is None` (call this on TRAIN); pass the
    returned encoder back in to transform val/test so the column set is
    identical across splits. `handle_unknown='ignore'` means a category unseen
    in train becomes an all-zero row instead of erroring (no leakage: the
    encoder's vocabulary is learned from train only).

    Returns (encoded_dataframe, fitted_encoder).
    """
    data = data.copy()
    if encoder is None:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                dtype=np.int8)
        encoder.fit(data[cols])
    ohe = pd.DataFrame(
        encoder.transform(data[cols]),
        columns=encoder.get_feature_names_out(cols),
        index=data.index,
    )
    return pd.concat([data.drop(columns=cols), ohe], axis=1), encoder


In [95]:
# Fit the encoder on TRAIN only, then reuse it to transform val/test so all
# three frames end up with identical one-hot columns (no leakage; unknown
# categories at transform time become all-zero rows).
train, ohe_encoder = one_hot_encode(train)
val, _ = one_hot_encode(val, encoder=ohe_encoder)
test, _ = one_hot_encode(test, encoder=ohe_encoder)

print('one-hot columns added:', len(ohe_encoder.get_feature_names_out(CATEGORICAL_COLS)))
print('col parity  train==val:', sorted(train.columns) == sorted(val.columns),
      ' val==test:', sorted(val.columns) == sorted(test.columns))
train.head()


one-hot columns added: 33
col parity  train==val: True  val==test: True


,week,sku,weekly_sales,feat_main_page,price,1week_sales_lag,2week_sales_lag,3week_sales_lag,4week_sales_lag,1week_revenue_lag,...,functionality_03.Bluetooth speakers,functionality_04.Selfie sticks,functionality_05.Bluetooth tracker,functionality_06.Mobile phone accessories,functionality_07.Headphones,functionality_08.Digital pencils,functionality_09.Smartphone stands,functionality_10.VR headset,functionality_11.Fitness trackers,functionality_12.Flash drives
4,2016-11-28,1,84.0,True,8.83,127.0,110.0,102.0,135.0,1050.29,...,0,0,0,1,0,0,0,0,0,0
5,2016-12-05,1,87.0,True,8.98,84.0,127.0,110.0,102.0,741.72,...,0,0,0,1,0,0,0,0,0,0
6,2016-12-12,1,64.0,True,10.40,87.0,84.0,127.0,110.0,781.26,...,0,0,0,1,0,0,0,0,0,0
7,2016-12-19,1,128.0,True,9.57,64.0,87.0,84.0,127.0,665.60,...,0,0,0,1,0,0,0,0,0,0
8,2016-12-26,1,23.0,True,10.49,128.0,64.0,87.0,84.0,1224.96,...,0,0,0,1,0,0,0,0,0,0


# 5. Modelling Matrix (X / y)

Build numeric `X` / `y` for each split. Target is `log_weekly_sales` (sales
are right-skewed and multiplicative); predictions are back-transformed with
`expm1` so metrics are in real sales units. Identifiers (`sku`, `week`, level
`weekly_sales`) are kept aside for per-SKU metrics and the Forecast contract.
`weeks_since_promo` NaN (no prior promo) is imputed with a large sentinel plus
a `never_promoted` flag so linear models work; trees are unaffected.

In [96]:
TARGET = 'log_weekly_sales'
DROP_NON_FEATURES = ['week', 'sku', 'weekly_sales', 'log_weekly_sales',
                     'feat_main_page', 'is_warmup', 'week_index']


def make_xy(df, target=TARGET):
    """Split an encoded frame into (X, y, ids).

    - y   : `target` (log sales).
    - X   : every column except identifiers / targets / pipeline helpers,
            fully numeric. `weeks_since_promo` NaN -> sentinel 999 plus a
            `never_promoted` flag (keeps linear models usable; trees handle
            either form).
    - ids : sku, week and level `weekly_sales` (y_true in real units) for
            per-SKU metrics and the forecast contract.

    Raw `sku` is intentionally NOT a feature: SKU character is already carried
    by the lag / rolling / price_vs_sku_mean features and the one-hot
    vendor/color/functionality columns, avoiding 44 sparse dummies and false
    ordinality for linear models. One-hot `sku` is an easy future ablation.
    """
    ids = df[['sku', 'week', 'weekly_sales']].reset_index(drop=True)
    y = df[target].reset_index(drop=True)
    X = df.drop(columns=DROP_NON_FEATURES).reset_index(drop=True)

    X['never_promoted'] = X['weeks_since_promo'].isna().astype(int)
    X['weeks_since_promo'] = X['weeks_since_promo'].fillna(999)

    assert X.select_dtypes(exclude='number').empty, \
        f"non-numeric feature cols: {list(X.select_dtypes(exclude='number').columns)}"
    assert X.isna().sum().sum() == 0, \
        f"NaN in: {X.columns[X.isna().any()].tolist()}"
    assert np.isfinite(X.to_numpy()).all(), "non-finite values in X"
    return X, y, ids


In [97]:
X_train, y_train, ids_train = make_xy(train)
X_val,   y_val,   ids_val   = make_xy(val)
X_test,  y_test,  ids_test  = make_xy(test)

assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), \
    "X column mismatch across splits"
print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")
print(f"{X_train.shape[1]} features, all numeric, 0 NaN")


X_train (3080, 71) | X_val (572, 71) | X_test (572, 71)
71 features, all numeric, 0 NaN


# 6. Baseline & Evaluation Metrics

Naive forecast = last week's sales (`1week_sales_lag`, already in real units).
All metrics (MAE, RMSE, MAPE) are computed in **real sales units**; model
log-predictions are `expm1`-back-transformed before scoring. MAPE skips weeks
with zero actual sales. Reported aggregate and per-SKU; naive is the floor
every model must beat.

In [98]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def regression_metrics(y_true, y_pred):
    """MAE, RMSE, MAPE (%) in the input units. MAPE ignores zero-actual rows
    (percentage error is undefined there)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    nz = y_true != 0
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE': np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100,
        'n': len(y_true),
    }


def per_sku_metrics(ids, y_true, y_pred):
    """Per-SKU metric table (sorted worst MAE first) for diagnostics."""
    d = ids[['sku']].copy()
    d['y_true'] = np.asarray(y_true, dtype=float)
    d['y_pred'] = np.asarray(y_pred, dtype=float)
    out = (d.groupby('sku', group_keys=False)
             .apply(lambda g: pd.Series(regression_metrics(g['y_true'], g['y_pred'])))
             .reset_index())
    return out.sort_values('MAE', ascending=False)


In [99]:
# Naive: predict last week's sales. 1week_sales_lag is already a feature and
# is in real units (not logged), so it is the level prediction directly.
naive_val = regression_metrics(ids_val['weekly_sales'],  X_val['1week_sales_lag'])
naive_test = regression_metrics(ids_test['weekly_sales'], X_test['1week_sales_lag'])
print('Naive  val :', {k: round(v, 3) for k, v in naive_val.items()})
print('Naive  test:', {k: round(v, 3) for k, v in naive_test.items()})


Naive  val : {'MAE': 47.851, 'RMSE': np.float64(217.34), 'MAPE': np.float64(108.686), 'n': 572}
Naive  test: {'MAE': 90.876, 'RMSE': np.float64(427.813), 'MAPE': np.float64(84.801), 'n': 572}


# 7. Model Selection & Training

A linear baseline (scaled **Ridge**) and a non-linear model
(**HistGradientBoostingRegressor** — within the 8 GB cap, fast, native NaN
handling, no extra dependencies). Both train on the log target. Models are
ranked on the **validation** set against the naive floor; the winner is
evaluated once on **test**, with worst-SKU diagnostics for the narrative team.

In [100]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

np.random.seed(42)

# Every model contributes real-unit predictions to these shared dicts; the
# final cell (section 9) turns them into one leaderboard. Naive = last week.
val_pred = {'Naive': X_val['1week_sales_lag'].to_numpy(dtype=float)}
test_pred = {'Naive': X_test['1week_sales_lag'].to_numpy(dtype=float)}
fitted = {}

base_models = {
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'HistGBR': HistGradientBoostingRegressor(
        random_state=42, learning_rate=0.08, max_iter=400, l2_regularization=1.0),
}
for name, model in base_models.items():
    model.fit(X_train, y_train)                        # trained on log target
    val_pred[name] = np.expm1(model.predict(X_val))    # back to real units
    test_pred[name] = np.expm1(model.predict(X_test))
    fitted[name] = model

print('base predictions ready:', list(val_pred))


base predictions ready: ['Naive', 'Ridge', 'HistGBR']


## 7.1 Hyperparameter Tuning (time-series CV)

Tune HistGBR with an **expanding-window, week-based** cross-validation built
only from the training period (no val/test leakage, panel-safe). Scoring is
real-unit MAE (`expm1` of the log target). The tuned model is added to the
prediction dicts as `HistGBR_tuned`.

In [101]:
from sklearn.model_selection import ParameterSampler


def time_series_folds(week_series, n_splits=4):
    """Expanding-window CV folds by calendar week (panel-safe).

    Returns a list of (train_idx, val_idx) positional arrays: each fold trains
    on all weeks up to a cut and validates on the next block of weeks, so no
    future week ever informs training. Works on panel data because the split
    is on weeks, not rows.
    """
    w = np.asarray(week_series)
    uw = np.array(sorted(pd.unique(w)))
    bounds = np.linspace(len(uw) // 2, len(uw), n_splits + 1, dtype=int)
    folds = []
    for i in range(n_splits):
        tr_end = uw[bounds[i] - 1]
        va_weeks = uw[bounds[i]:bounds[i + 1]]
        if len(va_weeks) == 0:
            continue
        folds.append((np.where(w <= tr_end)[0],
                       np.where(np.isin(w, va_weeks))[0]))
    return folds


param_dist = {
    'learning_rate': [0.03, 0.05, 0.08, 0.1],
    'max_iter': [200, 400, 600],
    'max_leaf_nodes': [15, 31, 63],
    'min_samples_leaf': [20, 40],
    'l2_regularization': [0.0, 1.0, 5.0],
}
folds = time_series_folds(ids_train['week'], n_splits=4)
y_train_real = np.expm1(y_train.to_numpy())            # sales units for CV MAE

best_params, best_cv = None, np.inf
for params in ParameterSampler(param_dist, n_iter=10, random_state=42):
    maes = []
    for tr, va in folds:
        m = HistGradientBoostingRegressor(random_state=42, **params)
        m.fit(X_train.iloc[tr], y_train.iloc[tr])
        pred = np.expm1(m.predict(X_train.iloc[va]))
        maes.append(mean_absolute_error(y_train_real[va], pred))
    cv = float(np.mean(maes))
    if cv < best_cv:
        best_params, best_cv = params, cv

print(f'best CV MAE: {best_cv:.3f}')
print('best params:', best_params)

hgb_tuned = HistGradientBoostingRegressor(random_state=42, **best_params)
hgb_tuned.fit(X_train, y_train)
val_pred['HistGBR_tuned'] = np.expm1(hgb_tuned.predict(X_val))
test_pred['HistGBR_tuned'] = np.expm1(hgb_tuned.predict(X_test))
fitted['HistGBR_tuned'] = hgb_tuned


best CV MAE: 36.495
best params: {'min_samples_leaf': 40, 'max_leaf_nodes': 15, 'max_iter': 200, 'learning_rate': 0.03, 'l2_regularization': 1.0}


# 8. ARIMA (per-SKU univariate)

A classical univariate baseline: one ARIMA per SKU, fit on that SKU's
`log1p(weekly_sales)` history, order chosen by lowest AIC over a small grid,
forecasting its horizon. History is carried up to each split's start (val:
weeks &lt; val_start; test: weeks &lt; test_start). Falls back to last value for
too-short series; predictions clipped at 0 (sales can't be negative).

In [102]:
import warnings
from statsmodels.tsa.arima.model import ARIMA

ARIMA_ORDERS = [(1, 1, 1), (2, 1, 1), (1, 1, 0), (0, 1, 1), (2, 1, 2)]


def _arima_forecast(hist_log, steps):
    """Fit each candidate order on a log1p-sales history, keep the lowest-AIC
    fit, and forecast `steps` ahead. Falls back to the last observed value if
    every order fails (short / degenerate series)."""
    best = None
    for order in ARIMA_ORDERS:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                res = ARIMA(hist_log, order=order).fit()
            if best is None or res.aic < best[0]:
                best = (res.aic, res)
        except Exception:
            continue
    if best is None:
        return np.repeat(hist_log[-1], steps)
    return np.asarray(best[1].forecast(steps))


def arima_predictions(history_raw, target_ids):
    """Per-SKU ARIMA: fit on each SKU's log1p-sales history, forecast that
    SKU's weeks in `target_ids`. Returns real-unit predictions (>= 0) aligned
    to target_ids row order."""
    preds = pd.Series(index=target_ids.index, dtype=float)
    for sku, grp in target_ids.groupby('sku'):
        hist = (history_raw[history_raw['sku'] == sku]
                .sort_values('week')['weekly_sales'].to_numpy())
        rows = grp.sort_values('week').index
        if len(hist) < 6:
            fc = np.repeat(np.log1p(hist[-1]) if len(hist) else 0.0, len(rows))
        else:
            fc = _arima_forecast(np.log1p(hist), len(rows))
        preds.loc[rows] = np.clip(np.expm1(fc), 0, None)
    return preds.to_numpy()


# History carried up to each split's start (val_start / test_start from §3.7).
val_pred['ARIMA'] = arima_predictions(data[data['week'] < val_start], ids_val)
test_pred['ARIMA'] = arima_predictions(data[data['week'] < test_start], ids_test)
print('ARIMA val :',
      {k: round(v, 3) for k, v in
       regression_metrics(ids_val['weekly_sales'], val_pred['ARIMA']).items()})


ARIMA val : {'MAE': 48.673, 'RMSE': np.float64(242.559), 'MAPE': np.float64(62.252), 'n': 572}


# 9. Final Selection & Test Evaluation

All models (Naive, Ridge, HistGBR, HistGBR_tuned, ARIMA) are ranked on the
**validation** set; the winner gets a single evaluation on the held-out
**test** set, with worst-SKU diagnostics for the narrative team.

In [103]:
# Validation leaderboard over every model that contributed predictions.
val_results = {name: regression_metrics(ids_val['weekly_sales'], p)
               for name, p in val_pred.items()}
leaderboard = (pd.DataFrame(val_results).T[['MAE', 'RMSE', 'MAPE', 'n']]
                 .sort_values('MAE'))
print('Validation leaderboard (real sales units):')
print(leaderboard.round(3))

best_name = leaderboard.index[0]
print('\nSelected model:', best_name)

# Single, final evaluation of the winner on the held-out TEST set.
pred_test = test_pred[best_name]
test_metrics = regression_metrics(ids_test['weekly_sales'], pred_test)
print(f"\n{best_name} - TEST (real units):",
      {k: round(v, 3) for k, v in test_metrics.items()})
print(f"Naive      - TEST (real units):",
      {k: round(v, 3) for k, v in naive_test.items()})

worst = per_sku_metrics(ids_test, ids_test['weekly_sales'], pred_test).head(5)
print('\nWorst 5 SKUs by MAE on test:')
print(worst.round(2).to_string(index=False))


Validation leaderboard (real sales units):
                  MAE     RMSE     MAPE      n
HistGBR_tuned  37.355  177.866   44.293  572.0
HistGBR        37.531  170.114   44.489  572.0
Naive          47.851  217.340  108.686  572.0
ARIMA          48.673  242.559   62.252  572.0
Ridge          48.683  222.580   55.823  572.0

Selected model: HistGBR_tuned

HistGBR_tuned - TEST (real units): {'MAE': 60.66, 'RMSE': np.float64(325.922), 'MAPE': np.float64(58.569), 'n': 572}
Naive      - TEST (real units): {'MAE': 90.876, 'RMSE': np.float64(427.813), 'MAPE': np.float64(84.801), 'n': 572}

Worst 5 SKUs by MAE on test:
 sku     MAE    RMSE   MAPE    n
  25 1188.46 1782.55  46.78 13.0
  27  273.39  835.68  69.99 13.0
  30  241.48  542.67 599.71 13.0
  15  211.24  622.81  32.47 13.0
   7   85.38  187.62  42.44 13.0


# 10. Confidence Intervals (Phase 8)

Residual-based 80% and 95% intervals. Residuals are taken from the **selected
model's validation predictions (train-only fit)** in **log space** (so the
band is multiplicative and respects the skew), then the winner is **refit on
train+val** for the delivered point forecast. Coverage is sanity-checked on
test (≈80% / ≈95% of points should fall inside). The contract carries the 95%
band; 80% coverage is reported for the spec.

In [104]:
from sklearn.base import clone

sel = best_name  # winner from the validation leaderboard


def _interval_bounds(point_log, resid_log):
    """Real-unit (point, lo80, hi80, lo95, hi95) from a log-space point
    forecast and a log-space residual sample. Lower bounds clipped at 0."""
    q80 = np.quantile(resid_log, [0.10, 0.90])
    q95 = np.quantile(resid_log, [0.025, 0.975])
    pt = np.expm1(point_log)
    return (pt,
            np.clip(np.expm1(point_log + q80[0]), 0, None),
            np.expm1(point_log + q80[1]),
            np.clip(np.expm1(point_log + q95[0]), 0, None),
            np.expm1(point_log + q95[1]))


if sel in fitted:                                  # sklearn winner
    resid_log = y_val.to_numpy() - fitted[sel].predict(X_val)

    final_model = clone(fitted[sel])
    final_model.fit(pd.concat([X_train, X_val], ignore_index=True),
                    pd.concat([y_train, y_val], ignore_index=True))

    val_pt, val_lo80, val_hi80, val_lo95, val_hi95 = _interval_bounds(
        fitted[sel].predict(X_val), resid_log)                 # train-only fit
    test_pt, test_lo80, test_hi80, test_lo95, test_hi95 = _interval_bounds(
        final_model.predict(X_test), resid_log)                # refit train+val
else:                                              # Naive / ARIMA fallback
    resid = ids_val['weekly_sales'].to_numpy() - val_pred[sel]
    q80 = np.quantile(resid, [0.10, 0.90])
    q95 = np.quantile(resid, [0.025, 0.975])
    val_pt, test_pt = val_pred[sel], test_pred[sel]
    val_lo80, val_hi80 = np.clip(val_pt + q80[0], 0, None), val_pt + q80[1]
    val_lo95, val_hi95 = np.clip(val_pt + q95[0], 0, None), val_pt + q95[1]
    test_lo80, test_hi80 = np.clip(test_pt + q80[0], 0, None), test_pt + q80[1]
    test_lo95, test_hi95 = np.clip(test_pt + q95[0], 0, None), test_pt + q95[1]

yt = ids_test['weekly_sales'].to_numpy()
cov80 = np.mean((yt >= test_lo80) & (yt <= test_hi80)) * 100
cov95 = np.mean((yt >= test_lo95) & (yt <= test_hi95)) * 100
print(f"Selected model: {sel} (refit on train+val for the delivered forecast)")
print(f"Test coverage  80% interval: {cov80:.1f}%  (target ~80)")
print(f"Test coverage  95% interval: {cov95:.1f}%  (target ~95)")


Selected model: HistGBR_tuned (refit on train+val for the delivered forecast)
Test coverage  80% interval: 74.7%  (target ~80)
Test coverage  95% interval: 93.5%  (target ~95)


# 11. Forecast Contract Output (Phase 9)

Emit the `Forecast` contract exactly: `sku_id, period, y_true, y_pred,
y_lower, y_upper, model_name` (note `sku_id`/`period`, not `sku`/`week`).
Backtest only: val + test weeks with `y_true` filled. Saved to
`outputs/forecast.csv` for the dashboard / AI pod.

In [105]:
CONTRACT_COLS = ['sku_id', 'period', 'y_true', 'y_pred',
                 'y_lower', 'y_upper', 'model_name']


def contract_frame(ids, y_pred, y_lower, y_upper, model_name):
    """Build a Forecast-contract DataFrame for one split (y_true filled,
    backtest). 95% band is used for y_lower/y_upper per the contract."""
    return pd.DataFrame({
        'sku_id': ids['sku'].to_numpy(),
        'period': pd.to_datetime(ids['week']).dt.date.astype(str),
        'y_true': ids['weekly_sales'].to_numpy(dtype=float),
        'y_pred': np.asarray(y_pred, dtype=float),
        'y_lower': np.asarray(y_lower, dtype=float),
        'y_upper': np.asarray(y_upper, dtype=float),
        'model_name': model_name,
    })[CONTRACT_COLS]


forecast = pd.concat([
    contract_frame(ids_val,  val_pt,  val_lo95,  val_hi95,  sel),
    contract_frame(ids_test, test_pt, test_lo95, test_hi95, sel),
], ignore_index=True).sort_values(['sku_id', 'period']).reset_index(drop=True)

# Contract sanity checks
assert list(forecast.columns) == CONTRACT_COLS
assert forecast.isna().sum().sum() == 0, "nulls in backtest contract output"
assert (forecast['y_lower'] <= forecast['y_upper']).all(), "lower > upper"

OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
forecast_path = os.path.join(OUT_DIR, 'forecast.csv')
forecast.to_csv(forecast_path, index=False)
print(f"wrote {forecast_path}: {forecast.shape[0]} rows, model={sel}")
forecast.head()


wrote outputs/forecast.csv: 1144 rows, model=HistGBR_tuned


,sku_id,period,y_true,y_pred,y_lower,y_upper,model_name
0,1,2018-04-02,28.0,19.127360,6.630043,58.809655,HistGBR_tuned
1,1,2018-04-09,13.0,18.722009,6.476380,57.605132,HistGBR_tuned
2,1,2018-04-16,15.0,20.800494,7.264309,63.781474,HistGBR_tuned
3,1,2018-04-23,27.0,14.532978,4.888368,45.157176,HistGBR_tuned
4,1,2018-04-30,13.0,16.013381,5.449571,49.556281,HistGBR_tuned


# 12. Hand-off (Phase 10)

Export per-SKU test metrics for the AI Pod's narratives and a one-line
overall summary. Regeneration instructions live in
`ml/ml_forecasting/README.md`.

In [106]:
# Per-SKU metrics on TEST for the selected (refit) model -> narratives.
sku_metrics = per_sku_metrics(ids_test, ids_test['weekly_sales'], test_pt)
sku_metrics.insert(0, 'model_name', sel)
sku_metrics_path = os.path.join(OUT_DIR, 'per_sku_metrics.csv')
sku_metrics.to_csv(sku_metrics_path, index=False)

overall = regression_metrics(ids_test['weekly_sales'], test_pt)
summary = pd.DataFrame([{
    'model_name': sel,
    'MAE': overall['MAE'], 'RMSE': overall['RMSE'], 'MAPE': overall['MAPE'],
    'cov80': cov80, 'cov95': cov95, 'n_test': overall['n'],
}])
summary.to_csv(os.path.join(OUT_DIR, 'test_summary.csv'), index=False)

print(f"wrote {sku_metrics_path}: {len(sku_metrics)} SKUs")
print('Overall TEST:', {k: round(v, 3) for k, v in overall.items()})
print(f"Coverage  80%={cov80:.1f}%  95%={cov95:.1f}%")
sku_metrics.head()


wrote outputs/per_sku_metrics.csv: 44 SKUs
Overall TEST: {'MAE': 58.72, 'RMSE': np.float64(312.925), 'MAPE': np.float64(61.413), 'n': 572}
Coverage  80%=74.7%  95%=93.5%


,model_name,sku,MAE,RMSE,MAPE,n
24,HistGBR_tuned,25,1125.085550,1679.775490,45.028469,13.0
26,HistGBR_tuned,27,270.192181,836.365569,61.699207,13.0
29,HistGBR_tuned,30,240.215360,536.744601,687.536592,13.0
14,HistGBR_tuned,15,210.858648,620.647209,29.868714,13.0
6,HistGBR_tuned,7,85.143224,185.293697,45.699161,13.0
